# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a step-by-step guide to loading and analyzing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) utilizing the `mlcroissant` library.

### Dataset Source
The dataset is accessed through a Croissant schema JSON-LD file available at the following URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading

Let's load the dataset's Croissant schema and explore its metadata and content. The `mlcroissant` library helps parse the Croissant schema and access record sets and fields by their `@id`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display high-level metadata about the dataset
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}, Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview

Review all available record sets and fields (columns/questions) in the dataset along with their `@id`. Always use the `@id` field for referencing entities.


In [ ]:
# List all record set @ids, their names and short descriptions
print("Record sets available in the dataset:")
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"- record_set @id: {rs.id}\n  name: {rs.name}")


In [ ]:
# For each record set, print the fields (@id and name)
for rs in record_sets:
    print(f"\nFields for record_set @id: {rs.id} ({rs.name})")
    for field in rs.fields:
        print(f"  - field @id: {field.id:40} name: {field.name:30} type: {field.data_type}")

## 3. Data Extraction

Load data from a specific record set into a pandas DataFrame for analysis. You should use the `@id` of the record set as seen above. We'll demonstrate extraction for all record sets found in the schema.

In [ ]:
# List all record set @ids (use variable for flexibility)
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

print(f"Extracting data for {len(record_set_ids)} record set(s): {record_set_ids}\n")
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    print(f"Loaded {len(records)} records for record_set '{record_set_id}'")
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    if not df.empty:
        print(f"Columns for '{record_set_id}':\n  {', '.join(df.columns)}\n")

# For demonstration, pick the first record set with data
main_record_set_id = None
for rid, df in dataframes.items():
    if not df.empty:
        main_record_set_id = rid
        break

if main_record_set_id:
    print(f"First few rows of record set '{main_record_set_id}':")
    display(dataframes[main_record_set_id].head())
else:
    print("No non-empty record sets found.")

## 4. Exploratory Data Analysis (EDA)

Apply common preprocessing and data filtering steps. We'll:
- Select a numeric field (e.g., 'gad7_total_score' or similar quantitative symptom score field by `@id`)
- Filter rows based on a threshold
- Normalize the feature
- Group data by a key categorical field (e.g., `village` or `gender` by their `@id`)


In [ ]:
# You should replace these with the actual @id for your dataset fields found in section 2 above
if main_record_set_id:
    df = dataframes[main_record_set_id].copy()
    
    # Guess fields. You should look up correct @id strings from above code cell.
    # Common names based on survey conventions are used here as placeholders.
    numeric_field_id = None
    group_field_id = None
    for c in df.columns:
        if "gad7" in c or "phq9" in c or "psq" in c:
            numeric_field_id = c
            break
    # Try for group field (village, gender, etc)
    for c in df.columns:
        if "village" in c.lower() or "gender" in c.lower() or "marital" in c.lower():
            group_field_id = c
            break
    
    print(f"Using field for numeric analysis: {numeric_field_id}")
    print(f"Grouping field: {group_field_id}")

    # Drop NA for numeric analysis
    num_df = df
    if numeric_field_id:
        num_df = num_df[pd.to_numeric(num_df[numeric_field_id], errors='coerce').notna()].copy()
        num_df[numeric_field_id] = pd.to_numeric(num_df[numeric_field_id], errors='coerce')

        # Filter for high values (example: threshold = 10)
        threshold = 10
        filtered_df = num_df[num_df[numeric_field_id] > threshold]
        print(f"Records with {numeric_field_id} > {threshold}: {len(filtered_df)}\n")
        display(filtered_df[[numeric_field_id]].head())

        # Normalize the field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized field sample:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    else:
        print("No suitable numeric field found to analyze.")

    # Group by group_field_id
    if numeric_field_id and group_field_id:
        grouped_df = (
            filtered_df[[group_field_id, numeric_field_id]]
            .groupby(group_field_id)
            .mean()
            .sort_values(numeric_field_id, ascending=False)
        )
        print(f"Grouped result by '{group_field_id}':")
        display(grouped_df.head(10))

## 5. Visualization

Visualize the distribution of the selected numeric field and group differences using matplotlib or seaborn. For example, show the histogram for 'gad7_total_score' (by `@id`) and a boxplot by village/gender if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(data=df, x=numeric_field_id, bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we successfully loaded the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library, explored its structure, extracted and analyzed its data, and visualized mental health assessment scores.

- We identified available record sets and fields by `@id`, maintaining robust references according to the Croissant standard.
- Performed simple preprocessing, normalization, and filtering on a core measurement field.
- Grouped and visualized numeric symptom scores by key categorical variables to gain insights into population mental health variability.

The dataset can now be used for further in-depth analysis on mental health status, assessment methodology, or demographic studies. For more advanced analytical workflows, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/api/python/) and [Croissant schema reference](https://mlcommons.org/croissant/1.0/).